In [ ]:
# ==============================================================================
# ⚙️  CONFIG — update these paths before running
# ==============================================================================
BASE_DIR         = "data/hdf5_data_final"
LM_PATH          = "checkpoints/4gram_big.arpa"


In [13]:
# Our phoneme mapping (from competition data)
LOGIT_TO_PHONEME = [
    'BLANK', 'AA', 'AE', 'AH', 'AO', 'AW',
    'AY', 'B', 'CH', 'D', 'DH',
    'EH', 'ER', 'EY', 'F', 'G',
    'HH', 'IH', 'IY', 'JH', 'K',
    'L', 'M', 'N', 'NG', 'OW',
    'OY', 'P', 'R', 'S', 'SH',
    'T', 'TH', 'UH', 'UW', 'V',
    'W', 'Y', 'Z', 'ZH', '|'
]

# Test with our Session 10 example
gt_ids = [10, 2, 31, 29, 40, 3, 40, 15, 33, 9, 40, 36, 13, 40, 31, 34, 40, 15, 11, 31, 40, 29, 31, 1, 28, 31, 17, 9, 40]
pred_ids = [10, 2, 31, 29, 40, 3, 40, 15, 33, 9, 40, 36, 13, 40, 9, 34, 40, 15, 11, 31, 40, 29, 31, 1, 28, 31, 17, 9, 40]

# Convert IDs → phoneme names
gt_phonemes = [LOGIT_TO_PHONEME[i] for i in gt_ids]
pred_phonemes = [LOGIT_TO_PHONEME[i] for i in pred_ids]

print("GT phonemes: ", " ".join(gt_phonemes))
print("Pred phonemes:", " ".join(pred_phonemes))

# Split at | (word boundaries)
def split_at_boundaries(phonemes):
    """Split phoneme list at | to get word-groups"""
    words = []
    current = []
    for p in phonemes:
        if p == '|':
            if current:
                words.append(tuple(current))
                current = []
        else:
            current.append(p)
    if current:
        words.append(tuple(current))
    return words

gt_words = split_at_boundaries(gt_phonemes)
pred_words = split_at_boundaries(pred_phonemes)

print("\n📊 GT word groups:")
for i, w in enumerate(gt_words):
    print(f"   Word {i+1}: {w}")

print(f"\n📊 Pred word groups:")
for i, w in enumerate(pred_words):
    print(f"   Word {i+1}: {w}")


GT phonemes:  DH AE T S | AH | G UH D | W EY | T UW | G EH T | S T AA R T IH D |
Pred phonemes: DH AE T S | AH | G UH D | W EY | D UW | G EH T | S T AA R T IH D |

📊 GT word groups:
   Word 1: ('DH', 'AE', 'T', 'S')
   Word 2: ('AH',)
   Word 3: ('G', 'UH', 'D')
   Word 4: ('W', 'EY')
   Word 5: ('T', 'UW')
   Word 6: ('G', 'EH', 'T')
   Word 7: ('S', 'T', 'AA', 'R', 'T', 'IH', 'D')

📊 Pred word groups:
   Word 1: ('DH', 'AE', 'T', 'S')
   Word 2: ('AH',)
   Word 3: ('G', 'UH', 'D')
   Word 4: ('W', 'EY')
   Word 5: ('D', 'UW')
   Word 6: ('G', 'EH', 'T')
   Word 7: ('S', 'T', 'AA', 'R', 'T', 'IH', 'D')


In [14]:
from nltk.corpus import cmudict

# CMUdict has stress markers (AH0, AH1). We strip them to match our phonemes.
cmu = cmudict.dict()

# Build reverse lookup: (phoneme_tuple) → [list of words]
reverse_cmu = {}
for word, pronunciations in cmu.items():
    for pron in pronunciations:
        # Strip stress markers: 'AH0' → 'AH', 'IY1' → 'IY'
        stripped = tuple(p.rstrip('012') for p in pron)
        if stripped not in reverse_cmu:
            reverse_cmu[stripped] = []
        reverse_cmu[stripped].append(word)

print(f"✅ Reverse CMUdict built: {len(reverse_cmu):,} unique pronunciations")

# Test lookups
test_words = [
    ('DH', 'AE', 'T', 'S'),
    ('AH',),
    ('G', 'UH', 'D'),
    ('W', 'EY'),
    ('T', 'UW'),
    ('D', 'UW'),  # ← our model's error
    ('G', 'EH', 'T'),
    ('S', 'T', 'AA', 'R', 'T', 'IH', 'D'),
]

print("\n📖 Looking up our word groups:")
for phonemes in test_words:
    matches = reverse_cmu.get(phonemes, ["❌ NOT FOUND"])
    print(f"   {phonemes} → {matches}")


✅ Reverse CMUdict built: 113,923 unique pronunciations

📖 Looking up our word groups:
   ('DH', 'AE', 'T', 'S') → ["that's"]
   ('AH',) → ['a', 'uh', 'uhh']
   ('G', 'UH', 'D') → ['good', 'goode']
   ('W', 'EY') → ['way', 'waye', 'wei', 'weigh', 'wey', 'whey', 'wy']
   ('T', 'UW') → ['tew', 'thuy', 'to', 'too', 'tu', 'tue', 'two']
   ('D', 'UW') → ['deux', 'dew', 'do', 'doo', 'douwe', 'du', 'due']
   ('G', 'EH', 'T') → ['get', 'goette']
   ('S', 'T', 'AA', 'R', 'T', 'IH', 'D') → ['started']


In [15]:
def phonemes_to_sentence(phoneme_groups):
    """Convert list of phoneme tuples to English sentence"""
    words = []
    for group in phoneme_groups:
        matches = reverse_cmu.get(group, None)
        if matches:
            words.append(matches[0])  # Take first match for now
        else:
            words.append(f"<{'_'.join(group)}>")  # Unknown word
    return " ".join(words)

gt_sentence = phonemes_to_sentence(gt_words)
pred_sentence = phonemes_to_sentence(pred_words)

print(f"🧠 Brain thought:     \"{gt_sentence}\"")
print(f"🤖 Model predicted:   \"{pred_sentence}\"")
print(f"\n📊 Word-level comparison:")
for i, (g, p) in enumerate(zip(gt_sentence.split(), pred_sentence.split())):
    match = "✅" if g == p else "❌"
    print(f"   Word {i+1}: GT='{g}' | Pred='{p}' {match}")


🧠 Brain thought:     "that's a good way tew get started"
🤖 Model predicted:   "that's a good way deux get started"

📊 Word-level comparison:
   Word 1: GT='that's' | Pred='that's' ✅
   Word 2: GT='a' | Pred='a' ✅
   Word 3: GT='good' | Pred='good' ✅
   Word 4: GT='way' | Pred='way' ✅
   Word 5: GT='tew' | Pred='deux' ❌
   Word 6: GT='get' | Pred='get' ✅
   Word 7: GT='started' | Pred='started' ✅


In [16]:
import kenlm
from itertools import product

model_lm = kenlm.Model('checkpoints/4gram_big.arpa')

def phonemes_to_sentence_with_lm(phoneme_groups, lm, top_k=3):
    """Convert phonemes to English, using LM to pick the best word"""
    
    # Step 1: Get ALL candidate words for each position
    candidates_per_word = []
    for group in phoneme_groups:
        matches = reverse_cmu.get(group, None)
        if matches:
            candidates_per_word.append(matches[:top_k])  # Top 3 candidates
        else:
            candidates_per_word.append([f"<{'_'.join(group)}>"])
    
    # Step 2: Show candidates
    print("   Candidates per word:")
    for i, cands in enumerate(candidates_per_word):
        print(f"     Position {i+1}: {cands}")
    
    # Step 3: If sentence is short enough, try all combinations
    # For long sentences, use greedy LM scoring
    n_words = len(candidates_per_word)
    
    if n_words <= 10:  # Try all combos for short sentences
        best_sentence = None
        best_score = float('-inf')
        
        for combo in product(*candidates_per_word):
            sentence = " ".join(combo)
            score = lm.score(sentence)
            if score > best_score:
                best_score = score
                best_sentence = sentence
        
        return best_sentence, best_score
    else:
        # Greedy: pick best word left-to-right using LM context
        chosen = []
        for cands in candidates_per_word:
            best_word = cands[0]
            best_score = float('-inf')
            context = " ".join(chosen)
            for word in cands:
                test = f"{context} {word}".strip()
                score = lm.score(test)
                if score > best_score:
                    best_score = score
                    best_word = word
            chosen.append(best_word)
        return " ".join(chosen), lm.score(" ".join(chosen))

# Test on GT phonemes
print("🧠 GT phonemes:")
gt_sentence_lm, gt_score = phonemes_to_sentence_with_lm(gt_words, model_lm)
print(f"   → \"{gt_sentence_lm}\" (LM score: {gt_score:.2f})\n")

# Test on PREDICTED phonemes (with the error)
print("🤖 Predicted phonemes:")
pred_sentence_lm, pred_score = phonemes_to_sentence_with_lm(pred_words, model_lm)
print(f"   → \"{pred_sentence_lm}\" (LM score: {pred_score:.2f})")

# Compare
print(f"\n📊 Results:")
print(f"   Without LM: \"{pred_sentence}\"")
print(f"   With LM:    \"{pred_sentence_lm}\"")


🧠 GT phonemes:
   Candidates per word:
     Position 1: ["that's"]
     Position 2: ['a', 'uh', 'uhh']
     Position 3: ['good', 'goode']
     Position 4: ['way', 'waye', 'wei']
     Position 5: ['tew', 'thuy', 'to']
     Position 6: ['get', 'goette']
     Position 7: ['started']
   → "that's a good way to get started" (LM score: -11.30)

🤖 Predicted phonemes:
   Candidates per word:
     Position 1: ["that's"]
     Position 2: ['a', 'uh', 'uhh']
     Position 3: ['good', 'goode']
     Position 4: ['way', 'waye', 'wei']
     Position 5: ['deux', 'dew', 'do']
     Position 6: ['get', 'goette']
     Position 7: ['started']
   → "that's a good way do get started" (LM score: -17.20)

📊 Results:
   Without LM: "that's a good way deux get started"
   With LM:    "that's a good way do get started"


Loading the LM will be faster if you build a binary file.
Reading /root/4gram_big.arpa
----5---10---15---20---25---30---35---40---45---50---55---60---65---70---75---80---85---90---95--100
****************************************************************************************************


In [57]:
def find_candidates_smart(phoneme_group, reverse_cmu, english_words):
    """Collect ALL exact + 1-edit English words, then let LM choose"""
    candidates = set()
    
    # 1. Exact English matches
    for w in reverse_cmu.get(phoneme_group, []):
        if w.lower() in english_words:
            candidates.add(w)
    
    # 2. ALL words 1 phoneme away (no limit!)
    target_len = len(phoneme_group)
    for pron, wrds in reverse_cmu.items():
        if abs(len(pron) - target_len) > 1:
            continue
        if phoneme_edit_distance(phoneme_group, pron) == 1:
            for w in wrds:
                if w.lower() in english_words:
                    candidates.add(w)
    
    return list(candidates) if candidates else ["_".join(phoneme_group)]

# Test
test = find_candidates_smart(('D', 'UW'), reverse_cmu, ENGLISH_WORDS)
print(f"Total candidates: {len(test)}")
print(f"'to' in candidates: {'to' in test}")
print(f"'do' in candidates: {'do' in test}")
print(f"Sample: {test[:15]}")


Total candidates: 92
'to' in candidates: True
'do' in candidates: True
Sample: ['d', 'wu', 'foo', 'suu', 'duke', 'who', 'lieu', 'douche', 'dough', 'adieu', 'nu', 'dee', 'lue', 'too', 'deuce']


In [61]:
# Fix: Both GT and Predictions use the same clean decoder
def phonemes_to_words(phoneme_ids, lm):
    """Convert phoneme IDs to English sentence using CMUdict + English filter + LM"""
    phonemes = [LOGIT_TO_PHONEME[i] for i in phoneme_ids]
    word_groups = split_at_boundaries(phonemes)
    
    candidates_per_word = []
    for group in word_groups:
        exact = reverse_cmu.get(group, [])
        english = [w for w in exact if w.lower() in ENGLISH_WORDS]
        if not english:
            english = exact[:1] if exact else ["_".join(group)]
        candidates_per_word.append(english[:3])
    
    from itertools import product
    best_sentence, best_score = None, float('-inf')
    
    n_combos = 1
    for c in candidates_per_word:
        n_combos *= len(c)
    
    if n_combos <= 50000:
        for combo in product(*candidates_per_word):
            sentence = " ".join(combo)
            score = lm.score(sentence)
            if score > best_score:
                best_score = score
                best_sentence = sentence
    else:
        chosen = []
        for cands in candidates_per_word:
            best_word, best_s = cands[0], float('-inf')
            for word in cands:
                s = lm.score(" ".join(chosen + [word]))
                if s > best_s:
                    best_s = s
                    best_word = word
            chosen.append(best_word)
        best_sentence = " ".join(chosen)
    
    return best_sentence

# Word-level edit distance for WER
def word_edit_distance(ref, hyp):
    ref_w = ref.split()
    hyp_w = hyp.split()
    n, m = len(ref_w), len(hyp_w)
    dp = [[0]*(m+1) for _ in range(n+1)]
    for i in range(n+1): dp[i][0] = i
    for j in range(m+1): dp[0][j] = j
    for i in range(1, n+1):
        for j in range(1, m+1):
            if ref_w[i-1] == hyp_w[j-1]: dp[i][j] = dp[i-1][j-1]
            else: dp[i][j] = min(dp[i-1][j]+1, dp[i][j-1]+1, dp[i-1][j-1]+1)
    return dp[n][m], len(ref_w)

# ===== FULL WER TEST =====
print("🔬 FULL WER TEST (Same decoder for GT and Predictions)\n")

total_errors, total_words = 0, 0
total_sentences, perfect = 0, 0

for s_idx in [0, 5, 10, 15, 20, 25, 30, 35, 40]:
    val_path = session_val_files[s_idx]
    if val_path is None: continue
    
    with h5py.File(val_path, "r") as f:
        keys = list(f.keys())[:3]
        for key in keys:
            neural = torch.tensor(f[key]["input_features"][:], dtype=torch.float32, device='cuda')
            seq_len = int(f[key].attrs["seq_len"])
            gt_ids = f[key]["seq_class_ids"][:seq_len].tolist()
            
            # Same decoder for both GT and prediction
            gt_sentence = phonemes_to_words(gt_ids, model_lm)
            pred_sentence = decode_brain_to_text(model, neural, s_idx)
            
            errors, n_words = word_edit_distance(gt_sentence, pred_sentence)
            total_errors += errors
            total_words += n_words
            total_sentences += 1
            if errors == 0: perfect += 1
            
            match = "✅" if errors == 0 else "❌"
            print(f"{match} S{s_idx}: GT=\"{gt_sentence}\" | Pred=\"{pred_sentence}\"")

wer = total_errors / total_words * 100
print(f"\n{'='*60}")
print(f"📊 RESULTS:")
print(f"   WER:              {wer:.1f}%")
print(f"   Perfect sentences: {perfect}/{total_sentences} ({perfect/total_sentences:.0%})")
print(f"   Total words:       {total_words}")
print(f"   Total errors:      {total_errors}")
print(f"{'='*60}")


🔬 FULL WER TEST (Same decoder for GT and Predictions)

✅ S0: GT="bring it closer" | Pred="bring it closer"
✅ S0: GT="they hope that" | Pred="they hope that"
✅ S0: GT="tell my family they are not hungry" | Pred="tell my family they are not hungry"
❌ S5: GT="u look down at your arm" | Pred="u look down at your harm"
❌ S5: GT="the bermuda triangle" | Pred="the P_ER_M_AH_T_IY T_R_AY_AH_L_AH_L"
❌ S5: GT="he's nine month's old" | Pred="he's dan mess old"
❌ S10: GT="that's a good way to get started" | Pred="that's a good way do get started"
❌ S10: GT="ai was on crutches" | Pred="ai was on T_R_AH_N_T_AH_S"
❌ S10: GT="when u stop to think of it" | Pred="when hugh staab to think of it"
❌ S15: GT="the birch canoe slid on the smooth planks" | Pred="the birch canoe cleaned on the S_P_K_UW_TH place"
❌ S15: GT="glue the sheet to the dark blue background" | Pred="clue the sheet to the D_AA_NG_K plew P_R_AE_K_G_R_ER_N"
❌ S15: GT="it's easy to tell the depth of a well" | Pred="it's easy to dell the T_AE